# Frontload-cl Colab smoke (1×A100)

Checks whether **one A100** can hold the real per-rank microbatch (`24×4096`) for OLMo2-370M with FlashAttention-2 and `torch.compile`.

This is **not** the platform 8×A100 run and does **not** exercise the primer/control curriculum or `s3://edullm-data`.

**Runtime → Change runtime type → GPU** (A100 if you have Pro). Then run all cells.

Repo path used below: `/content/OLMo-core` on the branch `edullm/frontload-cl`.

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime (Runtime → Change runtime type)"
props = torch.cuda.get_device_properties(0)
print(f"device: {props.name}")
print(f"memory: {props.total_memory / 1024**3:.1f} GiB")
print(f"capability: {props.major}.{props.minor}")
print(f"torch: {torch.__version__}  cuda: {torch.version.cuda}")
if props.major < 8:
    print("WARNING: FlashAttention-2 wants SM80+ (A100). Use --attn-backend torch on older GPUs.")

## 2. Clone the branch

Uses a public HTTPS clone. If the branch is not on GitHub yet, zip this repo from your laptop, upload to Colab, unzip to `/content/OLMo-core`, and skip the clone cell.

Private clone alternative:
```
from google.colab import userdata
token = userdata.get('GH_TOKEN')  # store a read-only PAT in Colab secrets
!git clone --depth 1 --branch edullm/frontload-cl https://{token}@github.com/edu-llm/OLMo-core.git /content/OLMo-core
```

In [ ]:
from pathlib import Path

REPO = Path("/content/OLMo-core")
BRANCH = "edullm/frontload-cl"
REMOTE = "https://github.com/edu-llm/OLMo-core.git"

if not (REPO / ".edullm" / "frontload_cl" / "colab_smoke.py").is_file():
    !git clone --depth 1 --branch {BRANCH} {REMOTE} {REPO}
else:
    print(f"already present: {REPO}")

%cd {REPO}
!git rev-parse --short HEAD
!ls .edullm/frontload_cl/colab_smoke.py

## 3. Install OLMo-core + FlashAttention-2 (match the platform)

Colab often ships a newer torch (you may see `2.11+cu128`) that has **no** official FA2 wheel. Without FA2, SDPA at the real microbatch (`24×4096`) typically **OOMs on a 40 GiB A100** — that is **not** evidence the platform 8×A100 shape fails (the image uses FA2).

This cell:
1. Pins **torch 2.9** (same major as the eduLLM image)
2. Installs OLMo-core editable (`--no-deps`)
3. Installs the **Dao FA 2.8.3 wheel** for torch2.9 / cu12 / cp312 (same pattern as `.edullm/Dockerfile`)

Expect several minutes for the torch pin. If FA2 still fails, stop — do not treat a torch-SDPA OOM as a platform result.

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *args])

import torch

print("Colab torch before pin:", torch.__version__, "cuda:", torch.version.cuda)

# Platform image pins torch 2.9; FA2 prebuilt wheels match that, not Colab's default 2.11.
if not torch.__version__.startswith("2.9."):
    print("Pinning torch==2.9.0+cu128 (and matching torchvision/torchaudio)…")
    pip(
        "torch==2.9.0+cu128",
        "torchvision==0.24.0+cu128",
        "torchaudio==2.9.0+cu128",
        "--index-url",
        "https://download.pytorch.org/whl/cu128",
    )
    print("Restart the runtime now (Runtime → Restart session), then re-run from cell 1.")
    raise SystemExit("torch pin installed; restart runtime, then continue from the GPU check")

# Editable install without the kitchen-sink `all` extra.
pip("-e", ".", "--no-deps")
pip("numpy", "rich", "cached-path", "safetensors", "dataclass-extensions", "bettermap", "pandas")

ATTN = "flash_2"
try:
    import flash_attn  # noqa: F401
    print("flash_attn already importable:", flash_attn.__version__)
except Exception:
    # Same URL pattern as .edullm/Dockerfile — never fall back to a source compile.
    abi = "TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE"
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheel = (
        "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/"
        f"flash_attn-2.8.3+cu12torch2.9cxx11abi{abi}-{py}-{py}-linux_x86_64.whl"
    )
    print("installing", wheel)
    try:
        pip(wheel, "--no-build-isolation", "--no-cache-dir")
        import flash_attn
        print("installed flash_attn", flash_attn.__version__)
    except Exception as exc:
        print("FA2 wheel install failed:", type(exc).__name__, exc)
        print(
            "Do NOT microbench with --attn-backend torch at 24×4096 on 40GiB — "
            "SDPA will OOM and that does not predict platform flash_2 memory."
        )
        ATTN = "unavailable"

print("ATTN_BACKEND =", ATTN)
assert ATTN == "flash_2", "Need flash_2 for a meaningful A100 microbench; fix the wheel install first"

## 4. Microbench (the important cell)

Same shape as one rank on `gpu-8xa100`: **24 sequences × 4096**. Success = a few steps complete without OOM; note `peak_mem_gib`.

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ATTN set in the install cell; override here if needed: ATTN = "torch"
!python .edullm/frontload_cl/colab_smoke.py gpu-info
!python .edullm/frontload_cl/colab_smoke.py microbench --steps 3 --attn-backend {ATTN}

## 5. Optional: synthetic data → short Trainer fit

Exercises composable data loader + `TransformerTrainModule` on a **flat** mix (not primer/control). Skip if the microbench already answered your question.

In [ ]:
!python .edullm/frontload_cl/colab_smoke.py write-data --out /content/frontload-synth
!python .edullm/frontload_cl/colab_smoke.py train --data /content/frontload-synth --steps 5 --attn-backend {ATTN}

## How to read the result

| Outcome | Meaning |
| --- | --- |
| `ok: true` with `attn_backend: flash_2`, peak mem well under ~40 GiB | Single-rank shape looks fine for A100 (same microbatch as one platform rank) |
| FA2 install failed / refused | Fix torch pin + FA wheel; **do not** treat SDPA OOM as a platform failure |
| CUDA OOM **with flash_2** | Real concern for the 8-GPU microbatch — investigate before full submit |
| CUDA OOM with `torch` SDPA | Expected on 40 GiB at 24×4096; not informative for the platform image |

Still needed later on the platform: 8-way HSDP/NCCL, real `frontload-cl-10b-v1`, and `.edullm/run-smoke.yaml`.